In [ ]:
import pandas as pd
import numpy as np
import holidays
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split


# Step 2: load only needed columns, dayfirst parsing


In [ ]:

# Step 2: load only needed columns, dayfirst parsing
df = pd.read_csv(
    r"Prod_data\tbl_device_history.csv",
    usecols=["trolleyId", "geozoneId", "createdOn", "status"],
    parse_dates=["createdOn"],
    dayfirst=True
)

# keep only status==1 (active) if relevant, drop geozone 0 (unknown)
df = df[(df["status"] == 1) & (df["geozoneId"] != 0)].copy()
df = df[["trolleyId", "geozoneId", "createdOn"]]


# Step 3: dedup - one record per trolley/zone/hour


In [ ]:

df = df.sort_values(["trolleyId", "geozoneId", "createdOn"])
df["date"] = df["createdOn"].dt.date
df["hour"] = df["createdOn"].dt.hour


In [ ]:
df = df.drop_duplicates(subset=["trolleyId", "geozoneId", "date", "hour"])


In [ ]:
agg = df.groupby(["date", "hour", "geozoneId"])["trolleyId"].nunique().reset_index()
agg.columns = ["date", "hour", "geozoneId", "trolley_count"]

In [ ]:

# Step 5: time features
agg['date'] = pd.to_datetime(agg['date'], errors='coerce')
agg["month"] = agg["date"].dt.month
agg["day"] = agg["date"].dt.day
agg["day_of_week"] = agg["date"].dt.dayofweek
agg["is_weekend"] = agg["date"].isin([5, 6]).astype(int)
agg["is_month_start"] = agg["date"].dt.is_month_start.astype(int)
agg["is_month_end"] = agg["date"].dt.is_month_end.astype(int)

# peak hour flags (adjust to your domain, e.g. 7-9 and 17-19)
agg["is_morning_peak"] = agg["hour"].between(7, 9).astype(int)
agg["is_evening_peak"] = agg["hour"].between(17, 19).astype(int)

# cyclical encodings
agg["hour_sin"] = np.sin(2 * np.pi * agg["hour"] / 24)
agg["hour_cos"] = np.cos(2 * np.pi * agg["hour"] / 24)
agg["month_sin"] = np.sin(2 * np.pi * agg["month"] / 12)
agg["month_cos"] = np.cos(2 * np.pi * agg["month"] / 12)
agg["dow_sin"] = np.sin(2 * np.pi * agg["day_of_week"] / 7)
agg["dow_cos"] = np.cos(2 * np.pi * agg["day_of_week"] / 7)


In [ ]:
my_holidays = holidays.Malaysia(years=range(agg["date"].dt.year.min(), agg["date"].dt.year.max() + 1))

In [ ]:

agg["is_holiday"] = agg["date"].apply(lambda d: 1 if d in my_holidays else 0)

# day before / after holiday (often affects traffic too)
holiday_dates = set(my_holidays.keys())
agg["is_day_before_holiday"] = agg["date"].apply(lambda d: 1 if (d + pd.Timedelta(days=3)) in holiday_dates else 0)
agg["is_day_after_holiday"] = agg["date"].apply(lambda d: 1 if (d - pd.Timedelta(days=3)) in holiday_dates else 0)


In [ ]:
# Step 6: time-based split (no shuffle)
full = full.sort_values("date").reset_index(drop=True)

full["day"] = full["date"].dt.day      # date-of-month (1-31)
full["month_num"] = full["date"].dt.month
full["year"] = full["date"].dt.year

cutoff = full["date"].max() - pd.Timedelta(days=14)
train = full[full["date"] <= cutoff]
test  = full[full["date"] > cutoff]

In [ ]:
X = full[["geozoneId", "hour", "day", "month_num", "year", "day_of_week", "is_weekend",
    "is_month_start", "is_month_end",
    "is_morning_peak", "is_evening_peak",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
    "is_holiday", "is_day_before_holiday", "is_day_after_holiday"]]
y = full[['trolley_count']]


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

model = DecisionTreeRegressor(
    max_depth=6,
    min_samples_leaf=20,
    random_state=42
)
model.fit(X_train, y_train)

preds = model.predict(X_test)
preds = np.clip(preds, 0, None)  # counts can't be negative

mae = mean_absolute_error(y_test, preds)
#rmse = mean_squared_error(y_test, preds, squared=False)
r2 = r2_score(y_test, preds)

print(f"MAE: {mae:.3f}")
#print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")